# 07 — AdaBoost baseline

This notebook evaluates one simple, untuned AdaBoost model on the same fixed five-fold splits used by the other baselines

AdaBoost combines weak decision trees sequentially, giving more attention to observations misclassified by earlier trees. It is CPU-friendly at this dataset size

AdaBoost treats `sii` as a multiclass target and does not use the ordinal distance between classes during training. Quadratic Weighted Kappa is still used for evaluation

## Imports and data loading

In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# Make src importable when the notebook is launched from notebooks/
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    ID_COLUMN,
    PROCESSED_DIR,
    RANDOM_STATE,
    RESULTS_DIR,
    TARGET,
    TRAIN_PATH,
)
from src.evaluation import create_cv_splits, evaluate_model
from src.features import build_features
from src.imputation import make_preprocessor

In [10]:
processed_train_path = PROCESSED_DIR / "train_features.parquet"

if processed_train_path.exists():
    train_features = pd.read_parquet(processed_train_path)
    print("Loaded Layer A features:", processed_train_path)
else:
    # Apply the same deterministic cleaning when notebook 06 has not been run yet
    train = pd.read_csv(TRAIN_PATH)
    train_features = build_features(train)
    print("Built Layer A features from:", TRAIN_PATH)

print("Feature table shape:", train_features.shape)

Built Layer A features from: /home/arina/Desktop/predict-internet-usage-ivanov-secret/data/train.csv
Feature table shape: (3960, 67)


## Dataset and fixed cross-validation splits

Only rows with a known target are used. `id` and `sii` are excluded from the model features. PCIAT leakage columns were already removed by Layer A

`create_cv_splits` reproduces the same stratified folds used by the baseline notebook, including stable fold membership after row reordering

In [11]:
labeled_train = (
    train_features[train_features[TARGET].notna()]
    .reset_index(drop=True)
)

feature_columns = [
    column
    for column in labeled_train.columns
    if column not in {ID_COLUMN, TARGET}
]

X = labeled_train[feature_columns].copy()
y = labeled_train[TARGET].astype(int)
ids = labeled_train[ID_COLUMN]

cv_splits = create_cv_splits(X=X, y=y, ids=ids)

print("X shape:", X.shape)
print("Target distribution:")
print(y.value_counts().sort_index())

X shape: (2736, 65)
Target distribution:
sii
0    1594
1     730
2     378
3      34
Name: count, dtype: int64


## AdaBoost model

A depth-1 decision tree is a deliberately weak learner, also called a decision stump. AdaBoost builds these trees sequentially and combines their votes

Median imputation and one-hot encoding are fit separately inside every training fold. Scaling is disabled because tree splits are unaffected by feature scale

In [12]:
adaboost_model = Pipeline(
    steps=[
        (
            "preprocessor",
            make_preprocessor(
                X,
                strategy="median",
                scale=False,  # Tree splits do not depend on feature scale
            ),
        ),
        (
            "model",
            AdaBoostClassifier(
                estimator=DecisionTreeClassifier(
                    max_depth=1,  # Keep each learner intentionally weak
                    random_state=RANDOM_STATE,
                ),
                n_estimators=50,  # Start with a small CPU-friendly ensemble
                learning_rate=1.0,  # Use sklearn's default update strength
                random_state=RANDOM_STATE,  # Reproduce estimator seeds
            ),
        ),
    ]
)

## Cross-validation evaluation

The shared evaluator reports training and validation QWK for every fold. A large gap between them indicates overfitting

In [13]:
adaboost_result = evaluate_model(
    model=adaboost_model,
    X=X,
    y=y,
    cv_splits=cv_splits,
)

Fold 1: training QWK=0.2600, validation QWK=0.2640
Fold 2: training QWK=0.3268, validation QWK=0.2798
Fold 3: training QWK=0.3665, validation QWK=0.3883
Fold 4: training QWK=0.3746, validation QWK=0.3475
Fold 5: training QWK=0.3144, validation QWK=0.2824
Mean training QWK: 0.3285
Mean validation QWK: 0.3124
Validation QWK standard deviation: 0.0475


## Save and compare results

The out-of-fold prediction counts show whether AdaBoost predicts every severity class or mostly follows the majority classes

In [14]:
training_scores = adaboost_result["training_scores"]
validation_scores = adaboost_result["validation_scores"]
oof_counts = np.bincount(
    adaboost_result["oof_predictions"],
    minlength=4,  # Include target classes with zero predictions
)

adaboost_results = pd.DataFrame(
    [
        {
            "model": "AdaBoostClassifier",
            **{
                f"fold_{fold_number}_qwk": score
                for fold_number, score in enumerate(
                    validation_scores,
                    start=1,
                )
            },
            "mean_training_qwk": np.mean(training_scores),
            "mean_validation_qwk": np.mean(validation_scores),
            "validation_std_qwk": np.std(validation_scores),
            **{
                f"oof_pred_{target_class}_count": count
                for target_class, count in enumerate(oof_counts)
            },
        }
    ]
).round(4)

RESULTS_DIR.mkdir(exist_ok=True)
results_path = RESULTS_DIR / "adaboost_cv_results.csv"
adaboost_results.to_csv(
    results_path,
    index=False,  # Keep the pandas row index out of the CSV
)

print("Saved results to:", results_path)
adaboost_results

Saved results to: /home/arina/Desktop/predict-internet-usage-ivanov-secret/results/adaboost_cv_results.csv


,model,fold_1_qwk,fold_2_qwk,fold_3_qwk,fold_4_qwk,fold_5_qwk,mean_training_qwk,mean_validation_qwk,validation_std_qwk,oof_pred_0_count,oof_pred_1_count,oof_pred_2_count,oof_pred_3_count
0,AdaBoostClassifier,0.264,0.2798,0.3883,0.3475,0.2824,0.3285,0.3124,0.0475,2054,486,196,0


In [15]:
baseline_results_path = RESULTS_DIR / "baseline_cv_results.csv"
baseline_results = pd.read_csv(baseline_results_path)

comparison_columns = [
    "model",
    "mean_training_qwk",
    "mean_validation_qwk",
    "validation_std_qwk",
]

comparison = pd.concat(
    [
        baseline_results[comparison_columns],
        adaboost_results[comparison_columns],
    ],
    ignore_index=True,
).sort_values("mean_validation_qwk", ascending=False)

comparison

,model,mean_training_qwk,mean_validation_qwk,validation_std_qwk
1,Ridge,0.4041,0.3677,0.0386
4,AdaBoostClassifier,0.3285,0.3124,0.0475
3,RandomForestClassifier,1.0000,0.2921,0.0249
2,DecisionTreeClassifier,1.0000,0.2209,0.0494
0,DummyClassifier,0.0000,0.0000,0.0000


## AdaBoost experiments

The following experiments change one aspect at a time and keep the data, preprocessing, metric, and CV splits fixed

In [16]:
def make_adaboost_model(
    max_depth,
    n_estimators,
    learning_rate,
    class_weight=None,
):
    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_preprocessor(
                    X,
                    strategy="median",
                    scale=False,  # Tree splits do not depend on feature scale
                ),
            ),
            (
                "model",
                AdaBoostClassifier(
                    estimator=DecisionTreeClassifier(
                        max_depth=max_depth,
                        class_weight=class_weight,
                        random_state=RANDOM_STATE,
                    ),
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def result_row(name, result, max_depth, n_estimators, learning_rate, class_weight):
    training_scores = result["training_scores"]
    validation_scores = result["validation_scores"]
    oof_counts = np.bincount(
        result["oof_predictions"],
        minlength=4,  # Include target classes with zero predictions
    )

    return {
        "experiment": name,
        "max_depth": max_depth,
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "class_weight": class_weight or "none",
        "mean_training_qwk": np.mean(training_scores),
        "mean_validation_qwk": np.mean(validation_scores),
        "validation_std_qwk": np.std(validation_scores),
        "train_validation_gap": (
            np.mean(training_scores) - np.mean(validation_scores)
        ),
        **{
            f"oof_pred_{target_class}_count": count
            for target_class, count in enumerate(oof_counts)
        },
    }

### Experiment 1: tree depth

Increasing `max_depth` lets each weak learner model more complex relationships. Depths 1, 2, and 3 are compared while the ensemble size and learning rate stay fixed

In [17]:
depth_results = {1: adaboost_result}

for max_depth in [2, 3]:
    print(f"\nAdaBoost | max_depth={max_depth}")
    model = make_adaboost_model(
        max_depth=max_depth,
        n_estimators=50,
        learning_rate=1.0,
    )
    depth_results[max_depth] = evaluate_model(
        model=model,
        X=X,
        y=y,
        cv_splits=cv_splits,
    )

depth_table = pd.DataFrame(
    [
        result_row(
            name=f"depth_{max_depth}",
            result=result,
            max_depth=max_depth,
            n_estimators=50,
            learning_rate=1.0,
            class_weight=None,
        )
        for max_depth, result in depth_results.items()
    ]
).round(4)

depth_table[
    [
        "max_depth",
        "mean_training_qwk",
        "mean_validation_qwk",
        "validation_std_qwk",
        "train_validation_gap",
    ]
]


AdaBoost | max_depth=2
Fold 1: training QWK=0.4182, validation QWK=0.3286
Fold 2: training QWK=0.3979, validation QWK=0.3739
Fold 3: training QWK=0.3922, validation QWK=0.3279
Fold 4: training QWK=0.4225, validation QWK=0.2929
Fold 5: training QWK=0.4456, validation QWK=0.3051
Mean training QWK: 0.4153
Mean validation QWK: 0.3257
Validation QWK standard deviation: 0.0277

AdaBoost | max_depth=3
Fold 1: training QWK=0.5013, validation QWK=0.2760
Fold 2: training QWK=0.5229, validation QWK=0.3373
Fold 3: training QWK=0.4530, validation QWK=0.3630
Fold 4: training QWK=0.4779, validation QWK=0.2162
Fold 5: training QWK=0.4785, validation QWK=0.2754
Mean training QWK: 0.4867
Mean validation QWK: 0.2936
Validation QWK standard deviation: 0.0517


,max_depth,mean_training_qwk,mean_validation_qwk,validation_std_qwk,train_validation_gap
0,1,0.3285,0.3124,0.0475,0.0160
1,2,0.4153,0.3257,0.0277,0.0896
2,3,0.4867,0.2936,0.0517,0.1931


**Conclusion:** depth 2 improves mean validation QWK from **0.3124** to **0.3257**. Depth 3 lowers it to **0.2936** and increases the train-validation gap, so depth 2 is selected

### Experiment 2: ensemble size and learning rate

More trees are paired with a smaller learning rate so each boosting step makes a more gradual correction. Tree depth remains fixed at 2

In [18]:
parameter_results = {
    (50, 1.0): depth_results[2],
}

for n_estimators, learning_rate in [(100, 0.5), (200, 0.2)]:
    print(
        f"\nAdaBoost | n_estimators={n_estimators}, "
        f"learning_rate={learning_rate}"
    )
    model = make_adaboost_model(
        max_depth=2,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
    )
    parameter_results[(n_estimators, learning_rate)] = evaluate_model(
        model=model,
        X=X,
        y=y,
        cv_splits=cv_splits,
    )

parameter_table = pd.DataFrame(
    [
        result_row(
            name=f"estimators_{n_estimators}_rate_{learning_rate}",
            result=result,
            max_depth=2,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            class_weight=None,
        )
        for (n_estimators, learning_rate), result in parameter_results.items()
    ]
).round(4)

parameter_table[
    [
        "n_estimators",
        "learning_rate",
        "mean_training_qwk",
        "mean_validation_qwk",
        "validation_std_qwk",
        "train_validation_gap",
    ]
]


AdaBoost | n_estimators=100, learning_rate=0.5
Fold 1: training QWK=0.4017, validation QWK=0.3477
Fold 2: training QWK=0.3844, validation QWK=0.3569
Fold 3: training QWK=0.3811, validation QWK=0.3876
Fold 4: training QWK=0.4131, validation QWK=0.2741
Fold 5: training QWK=0.4275, validation QWK=0.3542
Mean training QWK: 0.4016
Mean validation QWK: 0.3441
Validation QWK standard deviation: 0.0376

AdaBoost | n_estimators=200, learning_rate=0.2
Fold 1: training QWK=0.3965, validation QWK=0.3956
Fold 2: training QWK=0.3603, validation QWK=0.3550
Fold 3: training QWK=0.3707, validation QWK=0.3496
Fold 4: training QWK=0.4115, validation QWK=0.3047
Fold 5: training QWK=0.4168, validation QWK=0.3358
Mean training QWK: 0.3912
Mean validation QWK: 0.3481
Validation QWK standard deviation: 0.0295


,n_estimators,learning_rate,mean_training_qwk,mean_validation_qwk,validation_std_qwk,train_validation_gap
0,50,1.0,0.4153,0.3257,0.0277,0.0896
1,100,0.5,0.4016,0.3441,0.0376,0.0575
2,200,0.2,0.3912,0.3481,0.0295,0.0430


**Conclusion:** 200 trees with `learning_rate=0.2` perform best, reaching mean validation QWK **0.3481**. The smaller learning rate also reduces the train-validation gap

### Experiment 3: balanced class weights

Balanced weights give larger importance to rare target classes. This may improve rare-class recall, but it can also produce too many severe-class predictions

In [19]:
balanced_model = make_adaboost_model(
    max_depth=2,
    n_estimators=200,
    learning_rate=0.2,
    class_weight="balanced",
)

balanced_result = evaluate_model(
    model=balanced_model,
    X=X,
    y=y,
    cv_splits=cv_splits,
)

balance_table = pd.DataFrame(
    [
        result_row(
            name="unbalanced",
            result=parameter_results[(200, 0.2)],
            max_depth=2,
            n_estimators=200,
            learning_rate=0.2,
            class_weight=None,
        ),
        result_row(
            name="balanced",
            result=balanced_result,
            max_depth=2,
            n_estimators=200,
            learning_rate=0.2,
            class_weight="balanced",
        ),
    ]
).round(4)

balance_table[
    [
        "class_weight",
        "mean_validation_qwk",
        "oof_pred_0_count",
        "oof_pred_1_count",
        "oof_pred_2_count",
        "oof_pred_3_count",
    ]
]

Fold 1: training QWK=0.3511, validation QWK=0.3168
Fold 2: training QWK=0.3318, validation QWK=0.3621
Fold 3: training QWK=0.3581, validation QWK=0.3608
Fold 4: training QWK=0.3337, validation QWK=0.2836
Fold 5: training QWK=0.3673, validation QWK=0.3161
Mean training QWK: 0.3484
Mean validation QWK: 0.3279
Validation QWK standard deviation: 0.0299


,class_weight,mean_validation_qwk,oof_pred_0_count,oof_pred_1_count,oof_pred_2_count,oof_pred_3_count
0,none,0.3481,2026,568,141,1
1,balanced,0.3279,1595,228,642,271


**Conclusion:** balanced weights lower mean validation QWK from **0.3481** to **0.3279**. They increase class 3 predictions from 1 to 271 although class 3 has only 34 training examples, so balanced weights are not selected

## Final experiment comparison

In [20]:
experiment_results = pd.concat(
    [
        depth_table,
        parameter_table.iloc[1:],  # Depth 2 with 50 trees is already included
        balance_table.iloc[[1]],  # The unbalanced setup is already included
    ],
    ignore_index=True,
).sort_values("mean_validation_qwk", ascending=False)

experiment_results_path = (
    RESULTS_DIR / "adaboost_experiment_results.csv"
)
experiment_results.to_csv(
    experiment_results_path,
    index=False,  # Keep the pandas row index out of the CSV
)

print("Saved results to:", experiment_results_path)
experiment_results[
    [
        "experiment",
        "max_depth",
        "n_estimators",
        "learning_rate",
        "class_weight",
        "mean_training_qwk",
        "mean_validation_qwk",
        "validation_std_qwk",
    ]
]

Saved results to: /home/arina/Desktop/predict-internet-usage-ivanov-secret/results/adaboost_experiment_results.csv


,experiment,max_depth,n_estimators,learning_rate,class_weight,mean_training_qwk,mean_validation_qwk,validation_std_qwk
4,estimators_200_rate_0.2,2,200,0.2,none,0.3912,0.3481,0.0295
3,estimators_100_rate_0.5,2,100,0.5,none,0.4016,0.3441,0.0376
5,balanced,2,200,0.2,balanced,0.3484,0.3279,0.0299
1,depth_2,2,50,1.0,none,0.4153,0.3257,0.0277
0,depth_1,1,50,1.0,none,0.3285,0.3124,0.0475
2,depth_3,3,50,1.0,none,0.4867,0.2936,0.0517


## Final conclusion

The best AdaBoost setup uses depth 2, 200 estimators, `learning_rate=0.2`, and no class balancing. It improves mean validation QWK from **0.3124** to **0.3481**, but remains below Ridge at **0.3677**

These settings were selected using the same CV results shown here, so **0.3481 is an exploratory tuning score**, not a new unbiased performance estimate